In [ ]:
# =========================================================
# PREDICTIVE MAINTENANCE - RUL PREDICTION (FINAL VERSION)
# Hybrid CNN + BiLSTM (Clean + Correct + Stable)
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Bidirectional, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================================================
# 1. LOAD DATA
# =========================================================
train_path = 'train_FD001.txt'

cols = ['engine_id','cycle','op1','op2','op3'] + [f'sensor{i}' for i in range(1,22)]
train = pd.read_csv(train_path, sep=r'\s+', header=None)
train.columns = cols

# =========================================================
# 2. RUL CALCULATION
# =========================================================
rul = train.groupby('engine_id')['cycle'].max().reset_index()
rul.columns = ['engine_id', 'max_cycle']

train = train.merge(rul, on='engine_id')
train['RUL'] = train['max_cycle'] - train['cycle']

# IMPORTANT: RUL clipping
train['RUL'] = train['RUL'].clip(upper=125)

train.drop('max_cycle', axis=1, inplace=True)

# =========================================================
# 3. FEATURE SELECTION
# =========================================================
drop_sensors = ['sensor1','sensor5','sensor10','sensor16','sensor18','sensor19']
sensor_cols = [f'sensor{i}' for i in range(1,22) if f'sensor{i}' not in drop_sensors]

# =========================================================
# 4. NOISE REDUCTION (FIXED - ENGINE WISE)
# =========================================================
for col in sensor_cols:
    train[col] = train.groupby('engine_id')[col] \
                      .transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Handle missing values
train[sensor_cols] = train[sensor_cols].ffill().bfill()

# =========================================================
# 5. NORMALIZATION
# =========================================================
scaler = MinMaxScaler()
train[sensor_cols] = scaler.fit_transform(train[sensor_cols])

# =========================================================
# 6. SEQUENCE GENERATION-SLIDING WINDOW
# =========================================================
sequence_length = 30

def create_sequences(data, seq_length):
    X, y = [], []

    for engine in data['engine_id'].unique():
        temp = data[data['engine_id'] == engine]
        sensors = temp[sensor_cols].values
        rul = temp['RUL'].values

        for i in range(len(sensors) - seq_length):
            X.append(sensors[i:i+seq_length])
            y.append(rul[i+seq_length])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = create_sequences(train, sequence_length)

# Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# =========================================================
# 7. HYBRID CNN + BiLSTM MODEL
# =========================================================
hybrid_model = Sequential([
    Input(shape=(30, len(sensor_cols))),

    # CNN (spatial patterns)
    Conv1D(64, 3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(2),

    # BiLSTM (temporal patterns)
    Bidirectional(LSTM(64, activation='relu')),

    # Dense layers
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

hybrid_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# =========================================================
# 8. TRAINING
# =========================================================
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.0001)
]

print("Training model...")
history = hybrid_model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_val, y_val),
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# =========================================================
# 9. EVALUATION
# =========================================================
hybrid_pred = hybrid_model.predict(X_val).flatten()

def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    acc = np.mean(np.abs(y_true - y_pred) <= 10) * 100
    return mse, mae, rmse, r2, acc

mse, mae, rmse, r2, acc = evaluate(y_val, hybrid_pred)

print("\n--- FINAL RESULTS ---")
print(f"MSE  : {mse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")
print(f"Accuracy (±10 cycles): {acc:.2f}%")

Training model...
Epoch 1/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - loss: 1020.4882 - mae: 22.0221 - val_loss: 5723.1978 - val_mae: 65.9158 - learning_rate: 0.0010
Epoch 2/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 307.8918 - mae: 13.4448 - val_loss: 2216.1619 - val_mae: 37.8031 - learning_rate: 0.0010
Epoch 3/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 269.7552 - mae: 12.4431 - val_loss: 949.3873 - val_mae: 22.9914 - learning_rate: 0.0010
Epoch 4/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 240.4281 - mae: 11.7910 - val_loss: 185.6468 - val_mae: 9.8957 - learning_rate: 0.0010
Epoch 5/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 236.0848 - mae: 11.5230 - val_loss: 342.8942 - val_mae: 13.0813 - learning_rate: 0.0010
Epoch 6/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 213.2469 - mae: 10.9445 - val_loss: 179.5319 - val_mae: 10.3562 - learning_rate: 0.0010
Epoch 7/100
221/221 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 202.2192 - mae: 10.6788

In [3]:
# Flatten sequence data for ML models
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)

In [4]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train_flat, y_train)

lr_pred = lr_model.predict(X_val_flat)

In [5]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_flat, y_train)

rf_pred = rf_model.predict(X_val_flat)

In [6]:
from tensorflow.keras.layers import Flatten

cnn_model = Sequential([
    Input(shape=(30, len(sensor_cols))),
    Conv1D(64, 3, activation='relu'),
    MaxPooling1D(2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse')

cnn_model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), verbose=0)

cnn_pred = cnn_model.predict(X_val).flatten()

111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [ ]:
lstm_model = Sequential([
    Input(shape=(30, len(sensor_cols))),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')

lstm_model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), verbose=0)

lstm_pred = lstm_model.predict(X_val).flatten()

In [ ]:
results = {}

results['Linear Regression'] = evaluate(y_val, lr_pred)
results['Random Forest'] = evaluate(y_val, rf_pred)
results['CNN'] = evaluate(y_val, cnn_pred)
results['LSTM'] = evaluate(y_val, lstm_pred)
results['Hybrid (CNN+BiLSTM)'] = evaluate(y_val, hybrid_pred)

In [ ]:
print("\n--- MODEL COMPARISON ---")

for model, (mse, mae, rmse, r2, acc) in results.items():
    print(f"\n{model}")
    print(f"MSE  : {mse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"Acc  : {acc:.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Raw Results
# -----------------------------
models = ["Hybrid", "CNN", "LSTM", "RF", "LR"]

data = {
    "MSE":  [44.27, 189.15, 147.10, 191.47, 294.24],
    "MAE":  [4.96, 10.87, 8.31, 10.53, 14.14],
    "RMSE": [6.65, 13.75, 12.13, 13.84, 17.15],
    "R2":   [0.97, 0.89, 0.92, 0.89, 0.83],
    "Acc":  [87.78, 53.81, 69.75, 57.53, 39.89]
}

metrics = list(data.keys())

# -----------------------------
# 2. Normalize (IMPORTANT)
# -----------------------------
normalized_data = {}

for metric in metrics:
    values = np.array(data[metric])

    if metric in ["MSE", "MAE", "RMSE"]:
        # Lower is better → invert
        norm = (values.max() - values) / (values.max() - values.min())
    else:
        # Higher is better
        norm = (values - values.min()) / (values.max() - values.min())

    normalized_data[metric] = norm

# -----------------------------
# 3. Radar Setup
# -----------------------------
labels = metrics
num_vars = len(labels)

angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # close loop

# -----------------------------
# 4. Plot
# -----------------------------
plt.figure(figsize=(8,8))
ax = plt.subplot(111, polar=True)

for i, model in enumerate(models):
    values = [normalized_data[m][i] for m in metrics]
    values += values[:1]

    ax.plot(angles, values, linewidth=2, label=model)
    ax.fill(angles, values, alpha=0.1)

# -----------------------------
# 5. Formatting (Publication Ready)
# -----------------------------
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=12)

ax.set_title("Model Performance Comparison (Normalized Radar Chart)",
             size=14, weight='bold', pad=20)

ax.grid(alpha=0.3)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('corrected_radar.png', dpi=300)
plt.show()

In [ ]:
def plot_metric(values, title, ylabel, filename=None):

    # Highlight best model
    if ylabel in ["MSE", "MAE", "RMSE"]:
        highlight_index = np.argmin(values)
    else:
        highlight_index = np.argmax(values)

    colors = ['#b0b0b0'] * len(models)
    colors[highlight_index] = '#1f77b4'   # blue highlight

    plt.figure(figsize=(6,4))
    bars = plt.bar(models, values, color=colors)

    plt.title(title, fontsize=12, fontweight='bold')
    plt.xlabel("Models")
    plt.ylabel(ylabel)

    plt.xticks(rotation=25)

    # Value labels (important for your design)
    for i, v in enumerate(values):
        plt.text(i, v, f"{v:.2f}", ha='center', va='bottom', fontsize=9)

    plt.grid(axis='y', linestyle='--', alpha=0.6)

    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=300)

    plt.show()

In [ ]:
import os
os.makedirs("data", exist_ok=True)

np.save("data/y_val.npy", y_val)
np.save("data/cnn_pred.npy", cnn_pred)
np.save("data/lstm_pred.npy", lstm_pred)
np.save("data/rf_pred.npy", rf_pred)
np.save("data/hybrid_pred.npy", hybrid_pred)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

save_dir = "/content/drive/MyDrive/RUL_Project/graphs"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# ==============================
# IMPORTS
# ==============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==============================
# LOAD DATA (Modify paths if needed)
# ==============================
# These should come from your notebooks
y_val = np.load("data/y_val.npy")

cnn_pred = np.load("data/cnn_pred.npy")
lstm_pred = np.load("data/lstm_pred.npy")
rf_pred = np.load("data/rf_pred.npy")
hybrid_pred = np.load("data/hybrid_pred.npy")

# ==============================
# METRIC FUNCTION
# ==============================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    acc = np.mean(np.abs(y_true - y_pred) <= 10) * 100
    return [mse, mae, rmse, r2, acc]

# ==============================
# METRIC TABLE (COMBINED)
# ==============================
metrics = pd.DataFrame({
    "Model": ["CNN", "LSTM", "Random Forest","Linear Regression", "Hybrid"],
    "MSE": [
        evaluate(y_val, cnn_pred)[0],
        evaluate(y_val, lstm_pred)[0],
        evaluate(y_val, rf_pred)[0],
        evaluate(y_val, lr_pred)[0],
        evaluate(y_val, hybrid_pred)[0],

    ],
    "MAE": [
        evaluate(y_val, cnn_pred)[1],
        evaluate(y_val, lstm_pred)[1],
        evaluate(y_val, rf_pred)[1],
        evaluate(y_val, lr_pred)[1],
        evaluate(y_val, hybrid_pred)[1],

    ],
    "RMSE": [
        evaluate(y_val, cnn_pred)[2],
        evaluate(y_val, lstm_pred)[2],
        evaluate(y_val, rf_pred)[2],
        evaluate(y_val, lr_pred)[2],
        evaluate(y_val, hybrid_pred)[2],


    ],
    "R2": [
        evaluate(y_val, cnn_pred)[3],
        evaluate(y_val, lstm_pred)[3],
        evaluate(y_val, rf_pred)[3],
        evaluate(y_val, lr_pred)[3],
        evaluate(y_val, hybrid_pred)[3]
    ],
    "Accuracy(±10)": [
        evaluate(y_val, cnn_pred)[4],
        evaluate(y_val, lstm_pred)[4],
        evaluate(y_val, rf_pred)[4],
        evaluate(y_val, lr_pred)[4],
        evaluate(y_val, hybrid_pred)[4]
    ]
})

print("\n=== COMBINED METRICS TABLE ===")
print(metrics)

# ==============================
# 1. RUL TREND (Hybrid vs CNN)
# ==============================
# =========================================================
# RUL DEGRADATION GRAPH (ENGINE-WISE) — FINAL VERSION
# =========================================================

import matplotlib.pyplot as plt

# Select engine
engine_id = 1

# Extract engine data
engine_data = train[train['engine_id'] == engine_id].reset_index(drop=True)

# IMPORTANT: use same sequence length
X_engine, y_engine = create_sequences(engine_data, sequence_length)

# Predictions
cnn_pred = cnn_model.predict(X_engine).flatten()
hybrid_pred = model.predict(X_engine).flatten()   # your hybrid model

# =========================================================
# PLOT
# =========================================================
plt.figure(figsize=(14,6))

# Actual RUL (bold black)
plt.plot(y_engine,
         color='black',
         linewidth=3,
         label='Actual RUL')

# CNN prediction (dashed red)
plt.plot(cnn_pred,
         color='red',
         linestyle='--',
         linewidth=2.5,
         label='CNN Prediction')

# Hybrid prediction (solid blue)
plt.plot(hybrid_pred,
         color='blue',
         linewidth=2.5,
         label='Proposed Hybrid (CNN + BiLSTM)')

# Labels
plt.title(f'Engine {engine_id} — RUL Degradation Trend Comparison',
          fontsize=16, fontweight='bold')

plt.xlabel('Time (Cycles)', fontsize=12)
plt.ylabel('Remaining Useful Life (RUL)', fontsize=12)

# Improve readability
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

# Highlight difference visually
plt.fill_between(range(len(y_engine)),
                 y_engine,
                 hybrid_pred,
                 color='blue',
                 alpha=0.1)

# Save high-quality image (for paper)
plt.savefig('rul_trend_hybrid_vs_cnn.png', dpi=300, bbox_inches='tight')

plt.show()
# ==============================
# 2. ERROR DISTRIBUTION (RF)
# ==============================
plt.figure(figsize=(10,6))

plt.hist(y_val - rf_pred, bins=50, alpha=0.5, label='RF', color='green')
plt.hist(y_val - hybrid_pred, bins=50, alpha=0.5, label='Hybrid', color='blue')

plt.title('Error Distribution Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Prediction Error')
plt.ylabel('Frequency')

plt.legend()
plt.grid(alpha=0.3)

plt.savefig(f"{save_dir}/error_distribution.png", dpi=300)
plt.show()

# ==============================
# 3. SCATTER (LSTM)
# ==============================
plt.figure(figsize=(8,8))

plt.scatter(y_val, lstm_pred, color='red', alpha=0.4, label='LSTM')
plt.scatter(y_val, hybrid_pred, color='blue', alpha=0.4, label='Hybrid')

max_val = max(y_val)
plt.plot([0, max_val], [0, max_val], 'k--', linewidth=2)

plt.xlabel('Actual RUL')
plt.ylabel('Predicted RUL')
plt.title('Prediction Accuracy: LSTM vs Hybrid', fontsize=14, fontweight='bold')

plt.legend()
plt.grid(alpha=0.3)

plt.savefig(f"{save_dir}/Scatter_lstm_vs_Hybrid.png", dpi=300)
plt.show()

# ==============================
# 4. ERROR VS TIME (CNN vs Hybrid)
# ==============================
cnn_error = np.abs(y_val - cnn_pred)
hybrid_error = np.abs(y_val - hybrid_pred)

plt.figure()
plt.plot(cnn_error[:200], label="CNN Error", linestyle='--')
plt.plot(hybrid_error[:200], label="Hybrid Error", linewidth=2)
plt.xlabel('Time')
plt.ylabel('Error')

plt.title("Error Over Time: CNN vs Hybrid")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(f"{save_dir}/Error_vs_Time.png", dpi=300)
plt.show()


# ==============================
# 5. SEPARATE MODEL COMPARISON BAR PLOTS
# ==============================
# ==============================
# SEPARATE METRIC-WISE COMPARISON (File 2 style)
# ==============================

def plot_metric(metric_name, save_dir):

    values = metrics[metric_name].values
    models = metrics["Model"].tolist()

    highlight_index = models.index("Hybrid")

    colors = ['#b0b0b0'] * len(models)
    colors[highlight_index] = '#1f77b4'

    plt.figure(figsize=(6,4))
    bars = plt.bar(models, values, color=colors)

    plt.title(f"{metric_name} Comparison", fontweight='bold')
    plt.xlabel("Models")
    plt.ylabel(metric_name)

    plt.xticks(rotation=25)

    # Value labels
    for i, v in enumerate(values):
        plt.text(i, v, f"{v:.2f}", ha='center', va='bottom')

    plt.grid(axis='y', linestyle='--', alpha=0.6)

    # ✅ Define filename properly
    filename = f"{metric_name}_comparison.png"
    filepath = f"{save_dir}/{filename}"

    plt.savefig(filepath, dpi=300, bbox_inches='tight')

    plt.show()
    plt.close()

    print("Saved:", filepath)

# Individual plots (same style as File 2)
plot_metric("MSE", save_dir)
plot_metric("MAE", save_dir)
plot_metric("RMSE", save_dir)
plot_metric("R2", save_dir)
plot_metric("Accuracy(±10)", save_dir)

In [ ]:
import joblib

# 1. Save the Keras Model
model.save("rul_model.h5")

# 2. Save the Scaler (Crucial for consistent API predictions)
# Assuming your scaler is named 'scaler'
joblib.dump(scaler, "scaler.joblib")

print("Model and Scaler exported successfully!")